# Fault Analysis System - Experiment 1
## Vision-based Vibration Analysis using GPT-4

### Setup and Dependencies

In [14]:
!pip install -q -r ../requirements.txt

In [15]:
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import os
import json
import base64
from pathlib import Path
from datetime import datetime

In [16]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
base_url = "https://openrouter.ai/api/v1"

In [17]:
client = OpenAI(api_key=api_key, base_url=base_url)

### 1. Severity Configuration

In [18]:
severity_config = {
    "critical_threshold_g": 5.0,
    "moderate_threshold_g": 3.0,
    "minor_threshold_g": 1.5,
    "max_frequency_khz": 3.0
}
config_path = "../Data/severity_config.json"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    json.dump(severity_config, f, indent=2)
print("Severity Configuration Created:")
print(json.dumps(severity_config, indent=2))

Severity Configuration Created:
{
  "critical_threshold_g": 5.0,
  "moderate_threshold_g": 3.0,
  "minor_threshold_g": 1.5,
  "max_frequency_khz": 3.0
}


### 2. File Organization Setup

In [19]:
def create_output_directory(equipment_id, equipment_type, date_str):
    base_path = f"../Data/Generated_Files/{equipment_id}_{date_str}"
    os.makedirs(f"{base_path}/Waveform", exist_ok=True)
    os.makedirs(f"{base_path}/Spectra", exist_ok=True)
    return base_path

test_dir = create_output_directory("Pump-1", "Pump", "2026-10-12")
print(f"Created output directory: {test_dir}")

Created output directory: ../Data/Generated_Files/Pump-1_2026-10-12


### 3. Title Parser

In [20]:
def parse_title(title_str):
    try:
        parts = title_str.rsplit(' ', 2)
        if len(parts) != 3:
            raise ValueError(f"Invalid title format: {title_str}")
        metadata_str, date_str, time_str = parts
        metadata = metadata_str.split('\\\\')
        return {
            'equipment_id': metadata[0],
            'equipment_type': metadata[1],
            'orientation': metadata[2],
            'plot_type': metadata[3],
            'date': date_str,
            'time': time_str
        }
    except Exception as e:
        print(f"Title parsing error: {e}")
        return None

test_title = "Pump-1\\Pump\\DEA\\Spectrum 12-10-2026 14:59:16"
parsed = parse_title(test_title)
print("Parsed Title:")
print(json.dumps(parsed, indent=2))

Title parsing error: list index out of range
Parsed Title:
null


### 4. Vision Analysis Workflow

In [21]:
def analyze_plot(image_path, model="gpt-4-turbo"):
    try:
        with open(image_path, "rb") as img:
            b64_image = base64.b64encode(img.read()).decode('utf-8')
        prompt = """Analyze this vibration plot to identify:
1. Mechanical faults (bearings, unbalance, misalignment, looseness)
2. Severity assessment: Critical (>5g), Moderate (3-5g), Minor (<3g)
3. Specific frequency ranges affected
4. Recommended corrective actions
5. Vibration reduction strategies"""
        response = client.chat.completions.create(
            model="openai/gpt-4-turbo",
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_image}"}}
                ]
            }],
            max_tokens=1500
        )
        return response.choices[0].message.content
    except FileNotFoundError:
        return f"Error: Image file not found at {image_path}"
    except Exception as e:
        return f"Error analyzing plot: {str(e)}"

print("analyze_plot function defined successfully")

analyze_plot function defined successfully


### 5. Report Generation - Text Format

In [22]:
def generate_text_report(equipment_id, analysis_results, output_path):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    report = f"""================================================================================
EQUIPMENT VIBRATION ANALYSIS REPORT
Equipment ID: {equipment_id}
Generated: {timestamp}
================================================================================
"""
    for orientation, analyses in analysis_results.items():
        report += f"\n[{orientation} Orientation]\n"
        report += "-" * 50 + "\n"
        for plot_type, analysis in analyses.items():
            report += f"\n{plot_type} Analysis:\n{analysis}\n"
    report += "\n" + "=" * 80 + "\n"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w') as f:
        f.write(report)
    return report

print("generate_text_report function defined successfully")

generate_text_report function defined successfully


### 6. Report Generation - HTML Format

In [ ]:
def generate_html_report(equipment_id, analysis_results, output_path):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    html_content = f"""<!DOCTYPE html>
<html><head><meta charset="UTF-8">
<title>Vibration Analysis Report - {equipment_id}</title>
<style>
body {{font-family: Arial, sans-serif; max-width: 900px; margin: 0 auto; padding: 20px;}}
.header {{background-color: #2c3e50; color: white; padding: 20px; border-radius: 5px;}}
.section {{background-color: white; padding: 20px; margin: 20px 0; border-left: 4px solid #3498db;}}
h1 {{margin: 0;}} h2 {{color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;}}
</style>
</head><body>
<div class="header"><h1>Vibration Analysis Report</h1>
<p><strong>Equipment ID:</strong> {equipment_id}</p>
<div style="font-size: 14px;">Generated: {timestamp}</div></div>"""
    for orientation, analyses in analysis_results.items():
        html_content += f'<div class="section"><h2>{orientation} Orientation</h2>'
        for plot_type, analysis in analyses.items():
            html_content += f'<h3>{plot_type}</h3><pre>{analysis}</pre>'
        html_content += '</div>'
    html_content += '</body></html>'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w') as f:
        f.write(html_content)
    return html_content

print("generate_html_report function defined successfully")

### 7. Main Workflow Orchestrator

In [ ]:
def process_equipment_analysis(image_paths_dict, equipment_id, equipment_type, date_str):
    output_dir = create_output_directory(equipment_id, equipment_type, date_str)
    print(f"Processing {equipment_id}...\nOutput: {output_dir}")
    analysis_results = {}
    for orientation, plot_types in image_paths_dict.items():
        print(f"  Analyzing {orientation}...")
        analysis_results[orientation] = {}
        for plot_type, image_path in plot_types.items():
            if os.path.exists(image_path):
                print(f"    {plot_type}...", end="")
                analysis = analyze_plot(image_path)
                analysis_results[orientation][plot_type] = analysis
                print(" OK")
            else:
                analysis_results[orientation][plot_type] = "Image not available"
    report_dir = output_dir
    txt_path = f"{report_dir}/report.txt"
    generate_text_report(equipment_id, analysis_results, txt_path)
    print(f"  Text report: {txt_path}")
    html_path = f"{report_dir}/report.html"
    generate_html_report(equipment_id, analysis_results, html_path)
    print(f"  HTML report: {html_path}")
    return analysis_results, output_dir

print("process_equipment_analysis defined successfully")

### 8. Execution Example

In [ ]:
# image_paths = {
#     'DEA': {'Waveform': '../Data/plots/DEA_waveform.jpg', 'Spectrum': '../Data/plots/DEA_spectrum.jpg'},
#     'DEV': {'Waveform': '../Data/plots/DEV_waveform.jpg', 'Spectrum': '../Data/plots/DEV_spectrum.jpg'},
#     'NDE': {'Waveform': '../Data/plots/NDE_waveform.jpg', 'Spectrum': '../Data/plots/NDE_spectrum.jpg'}
# }
# results, output_path = process_equipment_analysis(image_paths, "Pump-1", "Pump", "2026-10-12")
print("Ready for analysis. Uncomment and update image paths to run.")

### 9. Vibration Reduction Strategies

In [ ]:
strategies = {
    "Precision Balancing": {"description": "Add counterweights using phase analysis", "tolerance": "<= 0.1g"},
    "Laser Alignment": {"description": "Maintain shaft alignment", "tolerance": "<= 0.05mm offset"},
    "Structural Reinforcement": {"methods": ["Epoxy grouting", "Steel gussets", "Foundation bolting"]},
    "Damping Solutions": {"methods": ["Tuned mass dampers", "Viscoelastic materials", "Isolation mounts"]}
}
print("Vibration Reduction Strategies:")
for strategy, details in strategies.items():
    print(f"\n{strategy}:")
    for key, value in details.items():
        print(f"  {key}: {value}")